### Flask Application for Nutrition Recommender App

In [ ]:
# Importing essential libraries
from flask import Flask, render_template, request, redirect, url_for, flash, session, redirect
import logging, re, os
from datetime import datetime
from werkzeug.security import generate_password_hash, check_password_hash 
import sqlite3
import csv
import math
import ast

from openpyxl import load_workbook
#from faker import Faker
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import string
import random
import csv

#Libraries for machine learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

# ------------------------------------
#Initialize Flask Application
app = Flask(__name__)
# ------------------------------------


# ------------------------------------
# In production set SECRET_KEY via environment variable
app.secret_key = os.environ.get("SECRET_KEY", "ftgongvsbn7283")

# ------------------------------------

# ------------------------------------
# Database path
DB_PATH = "patientdb.db"
#CSV_PATH = "pcos_data.csv"
# ------------------------------------

# ------------------------------------
def init_db():
   conn = sqlite3.connect(DB_PATH)
   conn.close()

# ------------------------------------
#Configure Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler("app.log"), logging.StreamHandler()],
)
log = logging.getLogger(__name__)
# ------------------------------------


# ------------------------------------
# Simple in-memory storage
REGISTERED_USERS = [] # each item: {"username", "email", "age", "created_at"}
# ------------------------------------

# ------------------------------------
# Validation patterns

##Ensure that first name is only letters and hyphens
FIRSTNAME_PATTERN = re.compile(r'^[A-Za-z-]{1,50}$')

##Ensure that last name is only letters and hyphens
LASTNAME_PATTERN = re.compile(r'[A-Za-z-]{1,50}$')

## Ensures the email has a basic valid structure of name@domain.tld
EMAIL_PATTERN = re.compile(r'^[\w\.-]+@[\w\.-]+\.[A-Za-z]{2,}$')

## Strong password pattern that requires lowercase, uppercase, digit, special character, and minimum 8 characters.
PASSWORD_PATTERN = re.compile(r'^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&]).{8,}$')
# ------------------------------------

# ------------------------------------
# Calculating and viewing user recommendations 

def user_recs(user_email, db_path="patientdb.db", top_n=10):
    #Establishing database connection
    conn = sqlite3.connect(db_path)

    #Query the database for the user's target nutrient vector
    user_query = """
        SELECT Target_Nutrient_Vector
        FROM patientdata
        WHERE lower(email) = lower(?)
    """
    cursor = conn.cursor()
    cursor.execute(user_query, (user_email,))
    user_row = cursor.fetchone()

    if not user_row:
        conn.close()
        return None
    
    #Retrieving patient nutrient vector
    #Converting the nutrient vector TEXT string into a real list
    patient_vector = user_row[0]
    patient_vector = ast.literal_eval(patient_vector)


    #Loading the saved food_matrix_5d table 
    food_df = pd.read_sql_query("SELECT * FROM food_matrix_5d", conn)

    #Renaming the Raw DQL nutrient column names
    food_df = food_df.rename(columns={
        "Fiber, total dietary": 'fiber_g',
        "Fatty acids, total polyunsaturated": 'pufa_g',
        "Magnesium, Mg": 'magnesium_mg',
        "Vitamin_D_Total_UG": 'vitamin_d_mcg',
        "Zinc, Zn": 'zinc_mg'
    })

    #Retrieving the names of foods to attach them to the matrix
    names_df = pd.read_sql_query("SELECT description AS food_description FROM food", conn)
    food_df['food_description'] = names_df['food_description']

    conn.close()

    #Isolating the 5 nutrients in the vector
    #For scaling
    nutrient_cols = ['fiber_g', 'pufa_g', 'magnesium_mg', 'vitamin_d_mcg', 'zinc_mg']
    
    #Mapping the nutrient values to the food matrix
    food_matrix = food_df[nutrient_cols].values
    patient_matrix = np.array(patient_vector).reshape(1, -1)

    #Scaling the nutrients
    scaler = MinMaxScaler()
    scaled_foods = scaler.fit_transform(food_matrix)
    scaled_patient = scaler.transform(patient_matrix)

    #Running the cosine similarity engine 
    similarity_scores = cosine_similarity(scaled_patient, scaled_foods)[0]
    food_results_df = food_df.copy()
    food_results_df['Match_Score'] = np.round(similarity_scores * 100, 1)

    #Returning the top N items as a list of dictionaries for Jinja2 HTML rendering
    top_foods = food_results_df.sort_values(by='Match_Score', ascending=False).head(top_n)
    return top_foods.to_dict(orient='records')
    

# ------------------------------------
# Routes for pages

# Route for the Home page
@app.route('/')
def home():
    return render_template('home.html')

# Route for the About page
@app.route('/about')
def about():
    return render_template('about.html')

# Route for the login page
@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == "GET":
        return render_template("login.html")
    
    # ----- Post request handling -----
    email = request.form.get("email", "").strip()
    password = request.form.get("password", "")

    if not (email and password):
        flash("Please enter email and password.")
        return redirect(url_for("login"))

    # ----- Database connection and Query -----
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    #Querying the user table to find a user matching the email
    cursor.execute("""
        SELECT first_name, last_name, email, password
        FROM patientdata
        WHERE lower(email) = lower(?)
    """, (email,))
    row = cursor.fetchone()
    conn.close() #Close connection immediately after fetching data

    # ----- Check if we found a user in the database
    if not row:
        flash("Invalid email or password.")
        return redirect(url_for("login"))
    
    #Extracting the user data from the database row
    db_first_name, db_last_name, db_email, db_password = row

    #Verifying password
    if password != db_password:
        flash("Invalid email or password.")
        return redirect(url_for("login"))

    #Saving user information in session upon successful login
    session["ID"] = db_email
    session["username"] = f"{db_first_name} {db_last_name}"
    return redirect(url_for("dashboard"))

# Route for the patient dashboard
@app.route("/dashboard")
def dashboard():
    if "ID" not in session:
        flash("Please log in to continue.")
        return redirect(url_for("login"))
    return render_template("dashboard.html")

# Route for the register page
@app.route('/register', methods =['GET', 'POST'])
def register():
    if request.method == 'POST':
        First_name = (request.form.get("first-name") or "").strip()
        Last_name = (request.form.get("surname") or "").strip()
        email    = (request.form.get("email") or "").strip()
        password = (request.form.get("password") or "").strip()
        
        try:

            ## ---- SERVER-SIDE INPUT VALIDATION FOR REGISTRATION FORM ---- ##

            # ----- Checking empty fields -----
             if not First_name:
                 raise ValueError("First name is required")
             if not Last_name:
                 raise ValueError("Last name is required")
             if not email:
                 raise ValueError("Email is required.")
             if not password:
                 raise ValueError("Password is required.")
             
            # ----- Type / Format checks -----
             if not FIRSTNAME_PATTERN.fullmatch(First_name):
                  raise ValueError("First name must only contain letters and hyphens")

             if not LASTNAME_PATTERN.fullmatch(Last_name):
                  raise ValueError("Last name must only contain letters and hyphens")
             
             if not EMAIL_PATTERN.fullmatch(email):
                  raise ValueError("Email format is invalid.")
             
             if not PASSWORD_PATTERN.fullmatch(password):
                  raise ValueError("Password format is invalid.")
             
             if len(email) > 254:
                  raise ValueError("Email too long.")

            # ----- Hashing password ------
             hashed_password = generate_password_hash(password, method="pbkdf2:sha256")

             conn= sqlite3.connect(DB_PATH)
             cursor = conn.cursor()

            # ----- Pre-check duplicate -----
             cursor.execute("SELECT 1 FROM patientdata WHERE lower(email) = lower(?)", (email,))
             if cursor.fetchone():
                 flash("This email is already registered. Please log in instead.")
                 conn.close()
                 return redirect(url_for("login"))
             
            # ----- Insert the user into the database ----- 
             try:
                 cursor.execute("""
                    INSERT INTO patientdata (first_name, last_name, email, password)
                    VALUES (?, ?, ?, ?)
                 """, (First_name, Last_name, email, hashed_password)) #doctor is set as default role for role based access 
                 conn.commit()
                 flash("Registration successful! Please log in.")
             except sqlite3.IntegrityError: 
                 flash("This email is already registered. Please log in.")
             finally:
                 conn.close()

             return redirect(url_for("success"))
        
        except ValueError as e:
            flash(str(e), 'error')
            log.warning("Validation failed: %s", e)
            return redirect(url_for("register"))
       
    return render_template("register.html")

# Route for Registration success page 
@app.route('/success')
def success():
    return render_template('register_success.html')

# Route for the view recommendations page
@app.route('/view_recs')
def view_recs():
    if "ID" not in session:
        flash("Please log in to view your recommendations.") 
        return redirect(url_for("login"))
    
    user_email = session["ID"]

    #Running the recommender engine
    top_10_foods = user_recs(user_email, DB_PATH, top_n=10)

    if top_10_foods is None:
        flash("Could not locate nutrient profile for your account.")
        return redirect(url_for("dashboard"))
    
    #Passing the recommendations results into the HTML page for display
    return render_template('view_recs.html', recommendations=top_10_foods, user_name=session.get("userFirstName", "User"))

if __name__ == "__main__":
    init_db()
    app.run(debug=False)  

 * Serving Flask app '__main__'
 * Debug mode: off


2026-07-11 18:22:48,219 [INFO] WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
2026-07-11 18:22:48,221 [INFO] Press CTRL+C to quit
